

## Introduction to Hugging Face

Hugging Face provides **state-of-the-art** NLP (and increasingly, multimodal) models and an easy-to-use Python library called **Transformers**. With just a few lines of code you can:

* **Load** pre-trained models for dozens of tasks
* **Run** inference via high-level “pipelines”
* **Fine-tune** on your own data

---

## 1. Setup in Google Colab

```bash
# Install the core libraries
!pip install transformers datasets huggingface_hub --quiet
```

> **Tip:** Colab often comes with a GPU—go to **Runtime → Change runtime type → GPU** for faster inference.

---

## 2. Quick-start with Pipelines

The `pipeline` API wraps tokenization, model loading, and inference in one object.

### 2.1 Sentiment Analysis

```python
from transformers import pipeline

# 1. Load a sentiment-analysis pipeline (defaults to a small model)
sentiment = pipeline("sentiment-analysis")

# 2. Run inference
examples = [
    "Hugging Face makes NLP super accessible!",
    "I dislike bugs in my code..."
]
results = sentiment(examples)
for text, res in zip(examples, results):
    print(f"{text!r:50} → label={res['label']}, score={res['score']:.3f}")
```

### 2.2 Text Generation

```python
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
prompt = "Once upon a time"
out = generator(prompt, max_length=30, num_return_sequences=1)
print(out[0]["generated_text"])
```




# Finetuning GPT-2

*In this lab you will work on finetuning GPT 2 to solve specific problems*

- https://www.kaggle.com/code/changyeop/how-to-fine-tune-gpt-2-for-beginners
- https://medium.com/@prashanth.ramanathan/fine-tuning-a-pre-trained-gpt-2-model-and-performing-inference-a-hands-on-guide-57c097a3b810


In [1]:

!pip install -q -U transformers datasets accelerate

from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if device == 0 else "CPU")

# 1) Sentiment analysis
sentiment = pipeline("sentiment-analysis", device=device)

examples = [
    "Hugging Face makes NLP super accessible!",
    "I dislike bugs in my code.",
    "This course is very useful and interesting."
]

results = sentiment(examples)
for text, res in zip(examples, results):
    print(f"{text!r:60} -> {res['label']}, score={res['score']:.3f}")

# 2) Text generation
generator = pipeline("text-generation", model="gpt2", device=device)

prompt = "Once upon a time"
out = generator(
    prompt,
    max_new_tokens=30,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.8,
    pad_token_id=generator.tokenizer.eos_token_id
)

print("\nGenerated text:")
print(out[0]["generated_text"])


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Device: GPU


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

'Hugging Face makes NLP super accessible!'                   -> POSITIVE, score=1.000
'I dislike bugs in my code.'                                 -> NEGATIVE, score=0.999
'This course is very useful and interesting.'                -> POSITIVE, score=1.000


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'pad_token_id', 'num_return_sequences', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanu


Generated text:
Once upon a time, many of us could be forgiven for thinking that a person's emotional state might be called an "experience" or "experiment". In fact


In [3]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    GPT2ForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import numpy as np
import evaluate
import torch

raw = load_dataset("stanfordnlp/imdb")

# Small Colab-friendly subset
train_ds = raw["train"].shuffle(seed=42).select(range(1000))
test_ds  = raw["test"].shuffle(seed=42).select(range(200))

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# GPT-2 has no pad token by default
tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
test_tok = test_ds.map(tokenize, batched=True, remove_columns=["text"])

model = GPT2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)
model.config.pad_token_id = tokenizer.pad_token_id

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

args = TrainingArguments(
    output_dir="./gpt2-imdb",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
print("Evaluation:", metrics)

# Save the fine-tuned model
trainer.save_model("./gpt2-imdb-final")
tokenizer.save_pretrained("./gpt2-imdb-final")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.624765,0.635158,0.875000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.624765,0.635158,1,0.875000


Evaluation: {'eval_loss': 0.6351575255393982, 'eval_accuracy': 0.875}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-imdb-final/tokenizer_config.json', './gpt2-imdb-final/tokenizer.json')

In [4]:
!pip install evaluate

In [8]:
import torch

from datasets import Dataset

from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)


sample_texts = [
    "Artificial intelligence helps organizations analyze large amounts of data.",
    "Machine learning models learn patterns from examples and use those patterns to make predictions.",
    "Natural language processing allows computers to process and generate human language.",
    "Data science combines statistics, programming, visualization, and machine learning.",
    "Responsible AI requires careful evaluation, transparency, privacy, and human oversight.",
] * 100

dataset = Dataset.from_dict({
    "text": sample_texts
})

dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

print("Training samples:", len(dataset["train"]))
print("Testing samples:", len(dataset["test"]))

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT-2 has no padding token by default
tokenizer.pad_token = tokenizer.eos_token
def tokenize_function(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )


tokenized = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print("Tokenization completed.")



model = GPT2LMHeadModel.from_pretrained("gpt2")

# Set padding token
model.config.pad_token_id = tokenizer.pad_token_id


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

args = TrainingArguments(
    output_dir="./gpt2-domain",

    num_train_epochs=1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    learning_rate=5e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=20,

    report_to="none",

    fp16=torch.cuda.is_available()
)


trainer = Trainer(
    model=model,

    args=args,

    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],

    # IMPORTANT:
    # Do NOT use tokenizer=tokenizer here.
    # New Transformers versions use processing_class.
    processing_class=tokenizer,

    data_collator=data_collator
)


print("\nStarting GPT-2 fine-tuning...\n")

trainer.train()

print("\nTraining completed!")

model_path = "./gpt2-domain-final"

trainer.save_model(model_path)

tokenizer.save_pretrained(model_path)

print("\nModel saved at:")
print(model_path)

prompt = "Artificial intelligence"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

generated = model.generate(

    **inputs,

    max_new_tokens=50,

    do_sample=True,

    temperature=0.8,

    top_p=0.95,

    pad_token_id=tokenizer.eos_token_id
)


generated_text = tokenizer.decode(
    generated[0],
    skip_special_tokens=True
)

print("\n" + "=" * 60)
print("GENERATED TEXT")
print("=" * 60)

print(generated_text)

print("=" * 60)

Training samples: 450
Testing samples: 50


Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenization completed.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss
1,0.054391,0.000076


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved at:
./gpt2-domain-final

GENERATED TEXT
Artificial intelligence helps organizations analyze large amounts of data. Learn more about the science of science.

Machine learning helps organizations analyze large amounts of data. Learn more about the science of science.

Machine learning helps organizations analyze large amounts of data. Learn more


In [10]:


import torch

from datasets import Dataset

from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

faq_pairs = [

    (
        "What is Python?",
        "Python is a high-level programming language widely used for software development, data science, and AI."
    ),

    (
        "What is machine learning?",
        "Machine learning is a method in which computers learn patterns from data to make predictions or decisions."
    ),

    (
        "What is NLP?",
        "Natural Language Processing is a field of AI that enables computers to process and generate human language."
    ),

    (
        "What is a transformer?",
        "A transformer is a neural-network architecture based on attention mechanisms and widely used for language tasks."
    ),

    (
        "What is fine-tuning?",
        "Fine-tuning adapts a pre-trained model to a specific task or domain using additional task-specific data."
    )

] * 100

texts = [
    f"Question: {question}\nAnswer: {answer}"
    for question, answer in faq_pairs
]


dataset = Dataset.from_dict({
    "text": texts
})


dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)


print("Training examples:", len(dataset["train"]))
print("Testing examples:", len(dataset["test"]))

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=160
    )


tokenized = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)
model = GPT2LMHeadModel.from_pretrained("gpt2")

model.config.pad_token_id = tokenizer.pad_token_id


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

args = TrainingArguments(

    output_dir="./gpt2-faq",

    num_train_epochs=1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    learning_rate=5e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=20,

    report_to="none",

    fp16=torch.cuda.is_available()
)


trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
    data_collator=data_collator
)

print("\nStarting FAQ model training...\n")

trainer.train()

print("\nFAQ model training completed!")


model_path = "./gpt2-faq-final"

trainer.save_model(model_path)

tokenizer.save_pretrained(model_path)

print("\nModel saved to:")
print(model_path)

prompt = (
    "Question: What is artificial intelligence?\n"
    "Answer:"
)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)


inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}


output = model.generate(

    **inputs,

    max_new_tokens=50,

    do_sample=True,

    temperature=0.7,

    top_p=0.9,

    pad_token_id=tokenizer.eos_token_id
)

answer = tokenizer.decode(
    output[0],
    skip_special_tokens=True
)

print("\n" + "=" * 60)
print("FAQ MODEL RESPONSE")
print("=" * 60)

print(answer)

print("=" * 60)

Training examples: 450
Testing examples: 50


Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


...

Epoch,Training Loss,Validation Loss
1,0.072767,0.057138


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


FAQ model training completed!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to:
./gpt2-faq-final

FAQ MODEL RESPONSE
Question: What is artificial intelligence?
Answer: Artificial intelligence is a method in which computers learn patterns from data to make predictions or decisions. AI is a method in which computers learn patterns from data to make predictions or decisions.
Answer: Artificial intelligence is a method in which computers learn patterns from data




# Use Case-Specific Problems Solvable by Fine-tuning GPT-2 on Custom Documents

Fine-tuning GPT-2 on domain-specific documents enables organizations to create specialized text generation models that address unique business challenges. This report examines several concrete applications where GPT-2 fine-tuning provides effective solutions to industry-specific problems by leveraging custom datasets.

## Domain-Specific Text Generation Applications

### Medical Query Response Systems

The healthcare industry faces significant challenges in providing accessible medical information to patients. Fine-tuning GPT-2 on medical datasets creates specialized models like MedGPT-2 that can address these issues effectively.

MedGPT-2 demonstrates how fine-tuned models can provide accurate responses to medical queries, symptom descriptions, and mental health questions[^5]. This application involves collecting diverse medical datasets from forums and medical websites to train a model capable of generating informative responses to patient questions[^5]. Such systems can reduce the burden on healthcare professionals by serving as a first-line information resource, particularly in underserved areas where medical expertise may be limited.

The implementation process requires careful data collection, model fine-tuning, and integration with user-friendly interfaces like Streamlit for practical deployment[^5]. Future enhancements could include expanding datasets and implementing feedback mechanisms to improve answer quality over time[^5].

### Financial Advisory Services

Financial institutions often struggle to provide personalized advice at scale. Fine-tuned GPT-2 models offer a solution by generating tailored financial guidance based on customer queries.

A financial advice GPT-2 model, fine-tuned on financial datasets, can assist users with budgeting, investment strategies, and financial planning[^7]. This application addresses the need for accessible financial guidance outside traditional advisor relationships. The model leverages the transformer-based architecture of GPT-2 while focusing specifically on financial contexts[^7].

Such models can be integrated into larger financial advisory systems, chatbots, or automated customer service platforms to enhance the financial literacy of customers while reducing operational costs for financial institutions[^7].

### Customer Service Automation

Customer service departments frequently handle repetitive queries that could be automated with appropriate natural language processing systems.

Banking customer service chatbots developed through GPT-2 fine-tuning demonstrate how pre-trained language models can be adapted to specific customer service domains[^6]. By fine-tuning models like distilGPT-2 on banking-related queries and responses, organizations can create conversational AI systems that handle routine customer inquiries effectively[^6].

The implementation process involves tokenizing and preparing datasets, utilizing Hugging Face's Transformers library, and creating user interfaces with tools like Gradio[^6]. These systems can significantly reduce response times and operational costs while maintaining consistent service quality.

### Legal Document Generation

Law firms and legal departments face challenges in drafting standardized legal opinions efficiently while maintaining quality and consistency.

Fine-tuning GPT-2 on legal opinion datasets enables the generation of technical legal reports that follow established formats and incorporate appropriate legal terminology[^4]. This application addresses the time-intensive nature of legal drafting while potentially reducing costs for clients.

Such systems could serve as first-draft generators for attorneys, who would then review and refine the machine-generated content. This workflow maintains quality control while accelerating document production.

## Technical Approaches to Fine-Tuning

### Hierarchical Domain Adaptation

When dealing with related but distinct domains, hierarchical fine-tuning approaches offer advantages over single-domain methods.

Research shows that representing domains hierarchically allows models to capture both domain-specific and general-domain information[^3]. By using adapter layers to represent nodes in a hierarchical structure, models can selectively share representations between related domains while avoiding negative transfer from unrelated domains[^3].

This approach is particularly valuable when fine-tuning for multiple related domains, such as hotel reviews and restaurant reviews, which share some linguistic features but retain domain-specific characteristics[^3]. The hierarchical structure allows upper nodes to be updated more frequently, encoding more general knowledge, while leaf nodes capture domain-specific information[^3].

### Implementation Considerations

Implementing GPT-2 fine-tuning for custom applications involves several critical steps that ensure successful model development.

The process typically begins with installing dependencies and importing necessary libraries, including PyTorch and Hugging Face Transformers[^2]. Data processing involves splitting datasets into training and validation sets (typically 90% for training, 10% for validation)[^2]. Tokenization using the GPT-2 tokenizer prepares input and target sequences with appropriate padding and truncation options[^2].

Training configurations must specify parameters like output directory, number of training epochs, batch size, evaluation strategy, and logging frequency[^2]. After fine-tuning, the model can be saved and deployed for inference, with consideration given to computational resources like GPU availability for optimal performance[^2].

## Application Development Process

### Medical Query Assistant Development

Creating specialized applications like medical query assistants follows a structured process from data collection through deployment.

The development workflow begins with gathering diverse datasets of medical queries and responses from appropriate sources[^5]. After preprocessing the data, the GPT-2 model is fine-tuned using the prepared dataset to adapt it specifically to medical terminology and question-answering patterns[^5].

Following successful fine-tuning, the model is integrated with front-end frameworks like Streamlit to create user-friendly interfaces where users can enter medical queries and receive generated responses[^5]. Backend integration connects the interface to the fine-tuned model, allowing for seamless question processing and answer generation[^5].

### Banking Customer Service Implementation

Building customer service chatbots for specific industries like banking follows similar patterns but with domain-specific considerations.

The development process involves fine-tuning pre-trained models like distilGPT-2 on banking-related conversations and FAQs[^6]. Web-based interfaces created with tools like Gradio allow users to interact with the model through intuitive chat interfaces[^6].

These implementations enable financial institutions to provide 24/7 customer service while maintaining consistent response quality and reducing operational costs[^6].

## Conclusion

Fine-tuning GPT-2 on custom documents presents viable solutions to domain-specific challenges across multiple industries. From healthcare to finance, legal services to customer support, the ability to adapt pre-trained language models to specialized domains creates opportunities for automation, improved service delivery, and cost reduction.

The success of these applications depends on carefully selected training data, appropriate fine-tuning methodologies, and thoughtful integration into existing workflows. As organizations continue to explore the potential of fine-tuned language models, we can expect to see increasingly sophisticated applications that address more complex domain-specific problems.

Future developments will likely focus on improving model accuracy, expanding datasets to cover more diverse scenarios, and implementing feedback mechanisms that allow fine-tuned models to improve continuously through real-world use.


[^1]: https://platform.openai.com/docs/guides/fine-tuning

[^2]: https://github.com/arham-kk/gpt2-finetune

[^3]: https://blog.allenai.org/efficient-hierarchical-domain-adaptation-using-pretrained-language-models-fdd04c001230?gi=0c070ef42bb7

[^4]: https://community.openai.com/t/is-fine-tuning-the-way-to-go-to-generate-legal-opinions-law-technical-reports/170169

[^5]: https://github.com/Saish459/Finetuning-LLMs-MedGPT

[^6]: https://www.youtube.com/watch?v=_Pndmc__-v4

[^7]: https://huggingface.co/RoamifyMML/financial-advice-gpt2

[^8]: https://github.com/MehwishFatimah/GPT2_Summarization

[^9]: https://www.restack.io/p/fine-tuning-answer-gpt2-cat-ai

[^10]: https://30dayscoding.com/blog/gpt-for-named-entity-recognition-and-entity-extraction

[^11]: https://www.youtube.com/watch?v=nsdCRVuprDY

[^12]: https://community.openai.com/t/fine-tuning-gpt2-with-gpt4-responses-is-it-allowed/326740

[^13]: https://github.com/lizatukhtina/fine-tune-gpt2-for-meetiing-summarization

[^14]: https://www.toptal.com/deep-learning/exploring-pre-trained-models

[^15]: https://www.cohorte.co/blog/fine-tuning-gpt-2-with-hugging-face-transformers-a-complete-guide

[^16]: https://www.linkedin.com/pulse/fine-tuning-gpt-2-large-language-model-unlocking-its-adamson-mbcs

[^17]: https://drlee.io/fine-tuning-gpt-2-for-sentiment-analysis-94ebdd7b5b24

[^18]: https://blog.devgenius.io/fine-tuning-the-gpt-2-large-language-model-unlocking-its-full-potential-66e3a082ab9c?gi=4773a5eba121

[^19]: https://fxis.ai/edu/how-to-fine-tune-the-gpt-2-model-for-specific-tasks/

[^20]: https://www.youtube.com/watch?v=2bqjzUX9ssE

[^21]: https://ar5iv.labs.arxiv.org/html/2112.08718

[^22]: https://fxis.ai/edu/how-to-fine-tune-the-gpt-2-model-a-comprehensive-guide-2/

[^23]: https://github.com/vaddhiparthy/GPT

[^24]: https://www.restack.io/p/transfer-learning-answer-gpt2-transfer-learning-cat-ai

[^25]: https://fxis.ai/edu/how-to-fine-tune-gpt-2-a-step-by-step-guide/

[^26]: https://fxis.ai/edu/how-to-fine-tune-a-gpt-2-model-on-a-custom-dataset/

[^27]: http://arxiv.org/pdf/2112.08718.pdf

[^28]: https://openai.com/index/fine-tuning-gpt-2/

[^29]: https://www.linkedin.com/pulse/part-2a-fine-tuning-gpt-2-harry-potter-language-generation-mehta-3m7lc

[^30]: https://itnext.io/easily-build-your-own-gpt-from-scratch-using-aws-51811b6355d3

[^31]: https://www.restack.io/p/fine-tuning-gpt-2-answer-techniques-cat-ai

[^32]: https://www.toolify.ai/ai-news/mastering-gpt2-finetuning-python-guide-for-llm-finetuning-588131

[^33]: https://www.linkedin.com/pulse/legal-drafting-ai-putting-gpt-2-practical-use-sergii-shcherbak

[^34]: https://ouci.dntb.gov.ua/en/works/4OoXnJ24/

[^35]: https://deepdesk.com/blog/gpt-and-customer-service

[^36]: https://huggingface.co/RoamifyMML/financial-advice-gpt2/blob/46c97bb80d0a6d886e30748fa20d7d3d42ec3f58/README.md

[^37]: https://www.youtube.com/watch?v=oEpLMb5D_G0

[^38]: https://scholars.lib.ntu.edu.tw/entities/publication/329d6f25-8778-4bd9-a9ab-7d9952e29d53

[^39]: https://www.linkedin.com/posts/shakti-pawar_fundamentals-finetuning-gpt2-on-medical-activity-7251144314393677825-9Zul

[^40]: https://www.linkedin.com/posts/manoj-ajjakana_building-a-customer-service-chatbot-with-activity-7241433202525937665-j0kV

[^41]: https://blog.aryansingh.space/blog/Fine-Tuning-GPT-2-for-Financial-News-and-Analysis-Text-Generation

[^42]: https://github.com/Jayveersinh-Raj/code_generation_gpt2

[^43]: https://www.intel.com/content/www/us/en/developer/articles/training/fine-tuning-gpt2-with-hugging-face-and-intel-gaudi.html

[^44]: https://github.com/aymanehachcham/GPT2_Text_Summarization

[^45]: https://sohaib.com/how-to-use-gpt2-for-question-answering/

[^46]: https://30dayscoding.com/blog/fine-tuning-gpt-for-relation-extraction-and-knowledge-base-population

[^47]: https://discuss.huggingface.co/t/fine-tuning-gpt2-for-question-answering/31895

[^48]: https://arxiv.org/pdf/2309.06112.pdf

[^49]: https://www.linkedin.com/pulse/crafting-coherent-contextually-relevant-text-gpt-2-technical-arora-anfoc

[^50]: https://github.com/omidiu/GPT-2-Fine-Tuning

[^51]: https://www.youtube.com/watch?v=C_pWDlZWNIE

[^52]: https://github.com/arham-kk/gpt2-finetune

[^53]: https://discuss.huggingface.co/t/how-to-train-gpt-2-for-text-summarization/31731

[^54]: https://blog.paperspace.com/generating-text-summaries-gpt-2/

[^55]: https://www.restack.io/p/fine-tune-gpt2-answer-summarization-cat-ai

[^56]: https://www.cohorte.co/blog/fine-tuning-gpt-2-with-hugging-face-transformers-a-complete-guide


# Problems

Below are practical problems that can be addressed by fine-tuning GPT-2 on publicly available datasets, along with a brief overview of how to solve each using Google Colab.

---

### 1. Sentiment Analysis on Movie Reviews

**Problem:**  
Automatically classify movie reviews as positive or negative.

**Public Data:**  
IMDb movie reviews dataset (available via Hugging Face Datasets library).

**How to Solve:**
- Load the IMDb dataset using `datasets.load_dataset("imdb")`[3][5].
- Preprocess and tokenize the reviews using GPT-2’s tokenizer.
- Fine-tune GPT-2 (or GPT-2 for sequence classification) for 2–3 epochs on a Colab GPU instance.
- Evaluate the model and use it to predict sentiment for new reviews[3][5].

---

### 2. Domain-Specific Text Generation (e.g., Harry Potter Fan Fiction)

**Problem:**  
Generate creative text in the style of a specific domain, such as Harry Potter fan fiction.

**Public Data:**  
Fan fiction or book excerpts from Project Gutenberg or fan sites.

**How to Solve:**
- Gather and clean a collection of relevant text (e.g., Harry Potter books or fan fiction)[4].
- Tokenize the text using GPT-2’s tokenizer.
- Fine-tune GPT-2 for language modeling (text generation) for several epochs.
- Use the model to generate new domain-specific stories or paragraphs[4].

---

### 3. FAQ Answer Generation for a Public Dataset

**Problem:**  
Automatically generate answers to frequently asked questions in a specific domain (e.g., COVID-19 FAQs, Stack Overflow programming questions).

**Public Data:**  
- COVID-19 FAQ datasets from official sources.
- Stack Overflow question-answer pairs (public data dumps).

**How to Solve:**
- Collect question-answer pairs and clean the dataset[1].
- Format data as prompt-response pairs for GPT-2.
- Tokenize and fine-tune GPT-2 on these pairs.
- Use the model to generate answers to new, similar questions[1].

---

### 4. Headline Generation for News Articles

**Problem:**  
Generate concise headlines for news articles.

**Public Data:**  
CNN/DailyMail news dataset or other open news datasets.

**How to Solve:**
- Download and preprocess article-headline pairs.
- Tokenize the data and fine-tune GPT-2 to generate headlines given article snippets.
- Evaluate model outputs on a validation set.

---

### 5. Poetry or Song Lyric Generation

**Problem:**  
Generate poetry or song lyrics in a particular style.

**Public Data:**  
Poetry Foundation corpus, Project Gutenberg poetry, or song lyrics datasets.

**How to Solve:**
- Gather and clean a corpus of poems or lyrics.
- Tokenize and fine-tune GPT-2 for text generation.
- Prompt the model with a first line or theme to generate new poetic content.

---

## General Steps for Fine-Tuning on Google Colab

1. **Set Up Colab and Enable GPU:**  
   Open a new Colab notebook, enable GPU support[2].

2. **Install Required Libraries:**  
   Install `transformers`, `torch`, and `datasets`[2][3][5].

3. **Load and Preprocess Data:**  
   Use Hugging Face Datasets or upload your own data. Clean and tokenize the text[1][3][5].

4. **Load Pre-trained GPT-2 Model:**  
   Use `GPT2LMHeadModel` for generation or `GPT2ForSequenceClassification` for classification tasks[3][5].

5. **Fine-Tune the Model:**  
   Use Hugging Face’s `Trainer` or a custom training loop. Set epochs and batch size to fit within Colab’s memory and time constraints[3][5].

6. **Evaluate and Save the Model:**  
   Test the model on a validation set and save it to Google Drive or download it[2][3].

---


Citations:
[1] https://www.restack.io/p/fine-tuning-answer-gpt-2-models-cat-ai
[2] https://www.reddit.com/r/learnmachinelearning/comments/1fm6gif/is_it_possible_to_finetune_pretrained_models_like/
[3] https://drlee.io/fine-tuning-gpt-2-for-sentiment-analysis-94ebdd7b5b24
[4] https://www.linkedin.com/pulse/part-2a-fine-tuning-gpt-2-harry-potter-language-generation-mehta-3m7lc
[5] https://www.cohorte.co/blog/fine-tuning-gpt-2-with-hugging-face-transformers-a-complete-guide
[6] https://labelyourdata.com/articles/gpt-fine-tuning
[7] https://customgpt.ai/openai-gpt-fine-tuning-cases/
[8] https://github.com/DivyanshTiwari20/fine-tunning-GPT2LMHeadModel
[9] https://drlee.io/fine-tuning-gpt-2-for-sentiment-analysis-94ebdd7b5b24
[10] https://www.datacamp.com/tutorial/fine-tuning-large-language-models
[11] https://www.restack.io/p/fine-tuning-answer-gpt-2-cat-ai
[12] https://www.linkedin.com/pulse/part-2a-fine-tuning-gpt-2-harry-potter-language-generation-mehta-3m7lc
[13] https://www.restack.io/p/fine-tuning-gpt-2-answer-techniques-cat-ai
[14] https://github.com/arham-kk/gpt2-finetune
[15] https://gist.github.com/MattPitlyk/45541145ad48b93da395f0a72ec2e7dc?short_path=2ad537e
[16] https://colab.research.google.com/github/philschmid/fine-tune-GPT-2/blob/master/Fine_tune_a_non_English_GPT_2_Model_with_Huggingface.ipynb
[17] https://www.youtube.com/watch?v=2bqjzUX9ssE
[18] https://platform.openai.com/docs/guides/fine-tuning
[19] https://github.com/openai/gpt-2/issues/155
[20] https://www.youtube.com/watch?v=nsdCRVuprDY
[21] https://www.toptal.com/deep-learning/exploring-pre-trained-models
[22] https://www.youtube.com/watch?v=C_pWDlZWNIE
[23] https://www.toolify.ai/gpts/level-up-your-nlp-finetune-llama-2-model-with-free-colab-326769
[24] https://colab.research.google.com/drive/13dZVYEOMhXhkXWfvSMVM1TTtUDrT6Aeh?usp=sharing
[25] https://www.kaggle.com/code/changyeop/how-to-fine-tune-gpt-2-for-beginners
[26] https://stackoverflow.com/questions/74712335/how-to-fine-tune-a-gpt-2-model



In [11]:

from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

# Load a general GPT-2 generator.
generator = pipeline(
    "text-generation",
    model="gpt2",
    device=device
)

# A) Headline-generation prototype
article = (
    "Researchers developed a new artificial intelligence system "
    "that helps students practice programming and receive immediate feedback."
)

headline_prompt = "Headline: "
headline = generator(
    headline_prompt + article,
    max_new_tokens=15,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.7,
    pad_token_id=generator.tokenizer.eos_token_id
)

print("Headline prototype:")
print(headline[0]["generated_text"])

# B) Poetry-generation prototype
poetry_prompt = "Write a short poem about learning artificial intelligence:\n"
poem = generator(
    poetry_prompt,
    max_new_tokens=60,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    pad_token_id=generator.tokenizer.eos_token_id
)

print("\nPoetry prototype:")
print(poem[0]["generated_text"])

# C) Simple qualitative evaluation checklist
print("\nEvaluation checklist:")
print("1. Relevance to the prompt")
print("2. Fluency and grammatical quality")
print("3. Domain correctness")
print("4. Repetition / hallucination")
print("5. Human evaluation on a held-out test set")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=15) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'top_p', 'num_return_sequences', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Headline prototype:
Headline: Researchers developed a new artificial intelligence system that helps students practice programming and receive immediate feedback.

It is based on a computer model that teaches students how to be

Poetry prototype:
Write a short poem about learning artificial intelligence:

I thought of doing it again to prove the point of this site. It was not very fun.

In fact, it made me feel terrible, because it made me feel like you, the reader, have to take your time and make the same point that I have and make that point

Evaluation checklist:
1. Relevance to the prompt
2. Fluency and grammatical quality
3. Domain correctness
4. Repetition / hallucination
5. Human evaluation on a held-out test set
